# ChemicalEntity ↔ Mutation Relation-Wise Merge

Merges Chemical–Mutation triples from CKG and EvoAGE; resolves chemical names via
PubChem, DrugBank (standard + extended); falls back to raw head value for any
remaining unresolved IDs; deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import pandas as pd
import numpy as np
import re

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/CHEMICALENTITY_MUTATION/ALL_CHEMICALENTITY_MUTATION.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

- **PubChem** — CID → IUPAC name and SMILES
- **DrugBank standard** — `drugbank_id` → name
- **DrugBank extended** — full `drugbank_all.csv` as fallback

In [2]:
Pubchem = pd.read_pickle(DB_DIR + 'pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem
print(f"PubChem IUPAC dict: {len(Pubchem_IUPAC_CID_Dict):,} entries")

PubChem IUPAC dict: 119,177,440 entries


In [17]:
Drugbank = pd.read_csv(DB_DIR + 'drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))

DB_All = pd.read_csv(DB_DIR + 'drugbank/drugbank_all.csv')
DB_All_dict = dict(zip(DB_All['drugbank_id'], DB_All['name']))

print(f"DrugBank dict:     {len(Drugbank_dict):,} entries")
print(f"DrugBank all dict: {len(DB_All_dict):,} entries")

DrugBank dict:     16,575 entries
DrugBank all dict: 16,575 entries


In [18]:
Drugbank_dict

{'DB00001': 'Lepirudin',
 'DB00002': 'Cetuximab',
 'DB00003': 'Dornase alfa',
 'DB00004': 'Denileukin diftitox',
 'DB00005': 'Etanercept',
 'DB00006': 'Bivalirudin',
 'DB00007': 'Leuprolide',
 'DB00008': 'Peginterferon alfa-2a',
 'DB00009': 'Alteplase',
 'DB00010': 'Sermorelin',
 'DB00011': 'Interferon alfa-n1',
 'DB00012': 'Darbepoetin alfa',
 'DB00013': 'Urokinase',
 'DB00014': 'Goserelin',
 'DB00015': 'Reteplase',
 'DB00016': 'Erythropoietin',
 'DB00017': 'Salmon calcitonin',
 'DB00018': 'Interferon alfa-n3',
 'DB00019': 'Pegfilgrastim',
 'DB00020': 'Sargramostim',
 'DB00022': 'Peginterferon alfa-2b',
 'DB00023': 'Asparaginase Escherichia coli',
 'DB00024': 'Thyrotropin alfa',
 'DB00025': 'Antihemophilic factor, human recombinant',
 'DB00026': 'Anakinra',
 'DB00027': 'Gramicidin D',
 'DB00028': 'Human immunoglobulin G',
 'DB00029': 'Anistreplase',
 'DB00030': 'Insulin human',
 'DB00031': 'Tenecteplase',
 'DB00032': 'Menotropins',
 'DB00033': 'Interferon gamma-1b',
 'DB00034': 'Inter

## 2. Load KG Sources

### 2a. CKG

In [4]:
CKG_Chemical_Mutation = pd.read_csv(PROC_DIR + 'CKG/File_2_Drug_MutationVariant_CKG.csv')
CKG_Chemical_Mutation.columns = CKG_Chemical_Mutation.columns.str.lower()
print(f"CKG:    {len(CKG_Chemical_Mutation):,} rows | columns: {list(CKG_Chemical_Mutation.columns)}")
CKG_Chemical_Mutation.rename(columns={'head_smiles': 'head_detail_name'}, inplace=True)

CKG_Chemical_Mutation['kg_type'] = 'Generalised'
CKG_Chemical_Mutation['species'] = 'Homo sapiens'
CKG_Chemical_Mutation['relation'] = 'ChemicalEntity_Mutation'
CKG_Chemical_Mutation['head_type'] = 'ChemicalEntity'
CKG_Chemical_Mutation['tail_type'] = 'Mutation'
CKG_Chemical_Mutation.head(2)

CKG:    13,661 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'source', 'kg_source', "head_alt_id's", 'head_iupac', 'head_smiles', 'head_id_is', 'tail_id_is', 'evidence', 'response', 'disease']


,head,relation,tail,head_type,relation_type,tail_type,source,kg_source,head_alt_id's,head_iupac,head_detail_name,head_id_is,tail_id_is,evidence,response,disease,kg_type,species
0,10184653,ChemicalEntity_Mutation,Q504U8_p.Gly719Ser,ChemicalEntity,curated,Mutation,CGI,CKG,DB08916,CN(C)C/C=C/C(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=C...,(E)-N-[4-(3-chloro-4-fluoroanilino)-7-[(3S)-ox...,Pubchem,protein_mutation_name,Early trials,Responsive,Lung,Generalised,Homo sapiens
1,DB00002,ChemicalEntity_Mutation,B6RC66_p.Gly465Arg,ChemicalEntity,curated,Mutation,CGI,CKG,DB00002,Cetuximab,NaN,Drugbank,protein_mutation_name,Pre-clinical,Resistant,Colorectal adenocarcinoma,Generalised,Homo sapiens


In [5]:
CKG_Chemical_Mutation_2 = pd.read_csv(PROC_DIR + 'CKG/File_3_Drug_MutationVariant_OncoKB_CKG.csv')
CKG_Chemical_Mutation_2.columns = CKG_Chemical_Mutation_2.columns.str.lower()
print(f"CKG:    {len(CKG_Chemical_Mutation_2):,} rows | columns: {list(CKG_Chemical_Mutation_2.columns)}")
CKG_Chemical_Mutation_2.rename(columns={'head_smiles': 'head_detail_name'}, inplace=True)

CKG_Chemical_Mutation_2['kg_type'] = 'Generalised'
CKG_Chemical_Mutation_2['species'] = 'Homo sapiens'

CKG_Chemical_Mutation_2['relation'] = 'ChemicalEntity_Mutation'
CKG_Chemical_Mutation_2['head_type'] = 'ChemicalEntity'
CKG_Chemical_Mutation_2['tail_type'] = 'Mutation'
CKG_Chemical_Mutation_2['species'] = 'Homo sapiens'

CKG_Chemical_Mutation_2.head(2)

CKG:    3,439 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'source', 'kg_source', "head_alt_id's", 'head_iupac', 'head_smiles', 'head_id_is', 'tail_id_is', 'evidence', 'response', 'disease']


,head,relation,tail,head_type,relation_type,tail_type,source,kg_source,head_alt_id's,head_iupac,head_detail_name,head_id_is,tail_id_is,evidence,response,disease,kg_type,species
0,10184653,ChemicalEntity_Mutation,F1JTL6_p.Thr790Met,ChemicalEntity,curated,Mutation,OncoKB,CKG,DB08916,CN(C)C/C=C/C(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=C...,(E)-N-[4-(3-chloro-4-fluoroanilino)-7-[(3S)-ox...,Pubchem,protein_mutation_name,R,1,Non-Small Cell Lung Cancer,Generalised,Homo sapiens
1,11167602,ChemicalEntity_Mutation,P10721-2_p.Thr670Ile,ChemicalEntity,curated,Mutation,OncoKB,CKG,DB08896,CNC(=O)C1=NC=CC(=C1)OC2=CC(=C(C=C2)NC(=O)NC3=C...,4-[4-[[4-chloro-3-(trifluoromethyl)phenyl]carb...,Pubchem,protein_mutation_name,approved,Responsive,Gastrointestinal Stromal Tumor,Generalised,Homo sapiens


### 2b. Hald

In [6]:
hald = pd.read_csv(PROC_DIR + 'hald/Chemical_Mutation_HALD.csv')
hald.rename(columns={'relation_source': 'kg_source', 'tail_name': 'head_detail_name'}, inplace=True)
hald.columns = hald.columns.str.lower()
print(f"EvoAGE: {len(hald):,} rows | columns: {list(hald.columns)}")

hald['kg_type'] = 'Aging'
hald['species'] = 'Homo sapiens'
hald.head(2)

EvoAGE: 29 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smiles_name', 'head_id_is']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_smiles_name,head_id_is,kg_type,species
0,Triglycerides,ChemicalEntity_Mutation,rs2805533,ChemicalEntity,associated,Mutation,HALD_KG,Triglycerides,Triglycerides,NaN,Aging,Homo sapiens
1,5793,ChemicalEntity_Mutation,rs1260326,ChemicalEntity,associated,Mutation,HALD_KG,"(3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O,Pubchem,Aging,Homo sapiens


## 3. Check and Fix Duplicate Columns

In [7]:
SOURCE_DFS = [
    ('CKG_Chemical_Mutation', CKG_Chemical_Mutation),
    ('CKG_Chemical_Mutation_2', CKG_Chemical_Mutation_2),
    ('hald',                hald),
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[CKG_Chemical_Mutation] ✓ no duplicates
[CKG_Chemical_Mutation_2] ✓ no duplicates
[hald] ✓ no duplicates


In [54]:
# # Edit keep='first'/'last' per DataFrame based on inspection above
# CKG_Chemical_Mutation = CKG_Chemical_Mutation.loc[:, ~CKG_Chemical_Mutation.columns.duplicated(keep='first')]
# EvoAGE                = hald.loc[:,                ~hald.columns.duplicated(keep='first')]

# for name, df in SOURCE_DFS:
#     remaining = df.columns[df.columns.duplicated()].tolist()
#     print(f"[{name}] {'✓ clean' if not remaining else '✗ still has dupes: ' + str(remaining)}")

## 4. Align Schemas and Concatenate

In [8]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 17,129 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,10184653,ChemicalEntity_Mutation,Q504U8_p.Gly719Ser,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,(E)-N-[4-(3-chloro-4-fluoroanilino)-7-[(3S)-ox...,NaN,Generalised,Homo sapiens
1,DB00002,ChemicalEntity_Mutation,B6RC66_p.Gly465Arg,ChemicalEntity,curated,Mutation,CKG,Drugbank,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens


In [9]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,14,14,28
tail_id_is,29,29,58
head_detail_name,5334,5334,10668


In [10]:
final_df[final_df['head_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
1,DB00002,ChemicalEntity_Mutation,B6RC66_p.Gly465Arg,ChemicalEntity,curated,Mutation,CKG,Drugbank,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens
2,allosteric AKT inhibitors,ChemicalEntity_Mutation,G3V4I6_p.Glu17Lys,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens
5,HSP90 inhibitors,ChemicalEntity_Mutation,S5LJ61_p.Arg175His,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens
10,EGFR inhibitor 1st gens,ChemicalEntity_Mutation,G9MC81_p.Asp761Tyr,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens
12,HSP90 inhibitors,ChemicalEntity_Mutation,L0EQY4_p.Arg175His,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16794,DB00002,ChemicalEntity_Mutation,F1CC75_p.Val600Glu,ChemicalEntity,curated,Mutation,CKG,Drugbank,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens
16802,DB01269,ChemicalEntity_Mutation,A0A2U3TZI2_p.Val600Glu,ChemicalEntity,curated,Mutation,CKG,Drugbank,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens
16820,DB00002,ChemicalEntity_Mutation,A0A0C4LC38_p.Val600Glu,ChemicalEntity,curated,Mutation,CKG,Drugbank,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens
16982,DB01269,ChemicalEntity_Mutation,A0A2R8Y679_p.Val600Glu,ChemicalEntity,curated,Mutation,CKG,Drugbank,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens


## 5. Resolve Head (Chemical) Names

1. PubChem IUPAC lookup (numeric CIDs).
2. Normalise DrugBank IDs to zero-padded 5-digit format.
3. DrugBank standard dict fallback for `DB`-prefixed IDs.
4. DrugBank extended dict (`drugbank_all`) second fallback.
5. Final fallback: use raw head value as name (covers compound names stored directly as IDs).
6. Drop any rows still missing `head_detail_name`.

In [11]:
# Step 1 – PubChem lookup
final_df['head_detail_name'] = final_df['head'].astype(str).map(Pubchem_IUPAC_CID_Dict)
final_df['head_smiles_name'] = final_df['head'].astype(str).map(Pubchem_CID_Smile_Dict)
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name
0,10184653,ChemicalEntity_Mutation,Q504U8_p.Gly719Ser,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,(E)-N-[4-(3-chloro-4-fluoroanilino)-7-[(3S)-ox...,NaN,Generalised,Homo sapiens,CN(C)C/C=C/C(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=C...
1,DB00002,ChemicalEntity_Mutation,B6RC66_p.Gly465Arg,ChemicalEntity,curated,Mutation,CKG,Drugbank,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens,NaN
2,allosteric AKT inhibitors,ChemicalEntity_Mutation,G3V4I6_p.Glu17Lys,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens,NaN
3,3062316,ChemicalEntity_Mutation,H7C4S5_p.Gly466Val,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,NaN,Generalised,Homo sapiens,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...
4,9915743,ChemicalEntity_Mutation,A0A0R9RWK2_p.Thr798Ile,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,(E)-N-[4-[3-chloro-4-(pyridin-2-ylmethoxy)anil...,NaN,Generalised,Homo sapiens,CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)OCC4=...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17124,Ceramides,ChemicalEntity_Mutation,rs183444295,ChemicalEntity,decreased,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN
17125,Ceramides,ChemicalEntity_Mutation,rs183444295,ChemicalEntity,tend,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN
17126,Linoleic Acids,ChemicalEntity_Mutation,rs174537,ChemicalEntity,associated,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN
17127,Linoleic Acids,ChemicalEntity_Mutation,rs953413,ChemicalEntity,associated,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN


In [19]:
# Step 2 – Normalise DrugBank IDs
def standardize_drugbank_id(val):
    if isinstance(val, str):
        m = re.match(r'^DB(\d+)$', val)
        if m:
            return 'DB' + m.group(1).zfill(5)
    return val

final_df['head'] = final_df['head'].apply(standardize_drugbank_id).astype(str)
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name
0,10184653,ChemicalEntity_Mutation,Q504U8_p.Gly719Ser,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,(E)-N-[4-(3-chloro-4-fluoroanilino)-7-[(3S)-ox...,NaN,Generalised,Homo sapiens,CN(C)C/C=C/C(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=C...
1,DB00002,ChemicalEntity_Mutation,B6RC66_p.Gly465Arg,ChemicalEntity,curated,Mutation,CKG,Drugbank,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens,NaN
2,allosteric AKT inhibitors,ChemicalEntity_Mutation,G3V4I6_p.Glu17Lys,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens,NaN
3,3062316,ChemicalEntity_Mutation,H7C4S5_p.Gly466Val,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,NaN,Generalised,Homo sapiens,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...
4,9915743,ChemicalEntity_Mutation,A0A0R9RWK2_p.Thr798Ile,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,(E)-N-[4-[3-chloro-4-(pyridin-2-ylmethoxy)anil...,NaN,Generalised,Homo sapiens,CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)OCC4=...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17124,Ceramides,ChemicalEntity_Mutation,rs183444295,ChemicalEntity,decreased,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN
17125,Ceramides,ChemicalEntity_Mutation,rs183444295,ChemicalEntity,tend,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN
17126,Linoleic Acids,ChemicalEntity_Mutation,rs174537,ChemicalEntity,associated,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN
17127,Linoleic Acids,ChemicalEntity_Mutation,rs953413,ChemicalEntity,associated,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN


In [20]:
# Step 3 – DrugBank fallback
mask = final_df['head_detail_name'].isna() & final_df['head'].str.startswith('DB')
final_df.loc[mask, 'head_detail_name'] = final_df.loc[mask, 'head'].map(Drugbank_dict)
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name
0,10184653,ChemicalEntity_Mutation,Q504U8_p.Gly719Ser,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,(E)-N-[4-(3-chloro-4-fluoroanilino)-7-[(3S)-ox...,NaN,Generalised,Homo sapiens,CN(C)C/C=C/C(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=C...
1,DB00002,ChemicalEntity_Mutation,B6RC66_p.Gly465Arg,ChemicalEntity,curated,Mutation,CKG,Drugbank,protein_mutation_name,Cetuximab,NaN,Generalised,Homo sapiens,NaN
2,allosteric AKT inhibitors,ChemicalEntity_Mutation,G3V4I6_p.Glu17Lys,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,NaN,NaN,Generalised,Homo sapiens,NaN
3,3062316,ChemicalEntity_Mutation,H7C4S5_p.Gly466Val,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,NaN,Generalised,Homo sapiens,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...
4,9915743,ChemicalEntity_Mutation,A0A0R9RWK2_p.Thr798Ile,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,(E)-N-[4-[3-chloro-4-(pyridin-2-ylmethoxy)anil...,NaN,Generalised,Homo sapiens,CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)OCC4=...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17124,Ceramides,ChemicalEntity_Mutation,rs183444295,ChemicalEntity,decreased,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN
17125,Ceramides,ChemicalEntity_Mutation,rs183444295,ChemicalEntity,tend,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN
17126,Linoleic Acids,ChemicalEntity_Mutation,rs174537,ChemicalEntity,associated,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN
17127,Linoleic Acids,ChemicalEntity_Mutation,rs953413,ChemicalEntity,associated,Mutation,HALD_KG,NaN,NaN,NaN,NaN,Aging,Homo sapiens,NaN


In [22]:

import numpy as np

final_df['head_detail_name'] = (
    final_df['head_detail_name']
    .replace(['NaN', 'nan', '', ' '], np.nan)
    .fillna(final_df['head'])
)


final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name
0,10184653,ChemicalEntity_Mutation,Q504U8_p.Gly719Ser,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,(E)-N-[4-(3-chloro-4-fluoroanilino)-7-[(3S)-ox...,NaN,Generalised,Homo sapiens,CN(C)C/C=C/C(=O)NC1=C(C=C2C(=C1)C(=NC=N2)NC3=C...
1,DB00002,ChemicalEntity_Mutation,B6RC66_p.Gly465Arg,ChemicalEntity,curated,Mutation,CKG,Drugbank,protein_mutation_name,Cetuximab,NaN,Generalised,Homo sapiens,NaN
2,allosteric AKT inhibitors,ChemicalEntity_Mutation,G3V4I6_p.Glu17Lys,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,Triglycerides,NaN,Generalised,Homo sapiens,NaN
3,3062316,ChemicalEntity_Mutation,H7C4S5_p.Gly466Val,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,NaN,Generalised,Homo sapiens,CC1=C(C(=CC=C1)Cl)NC(=O)C2=CN=C(S2)NC3=CC(=NC(...
4,9915743,ChemicalEntity_Mutation,A0A0R9RWK2_p.Thr798Ile,ChemicalEntity,curated,Mutation,CKG,Pubchem,protein_mutation_name,(E)-N-[4-[3-chloro-4-(pyridin-2-ylmethoxy)anil...,NaN,Generalised,Homo sapiens,CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)OCC4=...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17124,Ceramides,ChemicalEntity_Mutation,rs183444295,ChemicalEntity,decreased,Mutation,HALD_KG,NaN,NaN,Ceramides,NaN,Aging,Homo sapiens,NaN
17125,Ceramides,ChemicalEntity_Mutation,rs183444295,ChemicalEntity,tend,Mutation,HALD_KG,NaN,NaN,Ceramides,NaN,Aging,Homo sapiens,NaN
17126,Linoleic Acids,ChemicalEntity_Mutation,rs174537,ChemicalEntity,associated,Mutation,HALD_KG,NaN,NaN,Linoleic Acids,NaN,Aging,Homo sapiens,NaN
17127,Linoleic Acids,ChemicalEntity_Mutation,rs953413,ChemicalEntity,associated,Mutation,HALD_KG,NaN,NaN,Linoleic Acids,NaN,Aging,Homo sapiens,NaN


In [23]:
# Step 4 – Assign head_id_is
final_df['head_id_is'] = np.where(
    final_df['head'].isna(), None,
    np.where(final_df['head'].str.startswith('DB'), 'Drugbank', 'Pubchem')
)
final_df
# Step 5 – Drop unresolvable rows
before = len(final_df)
final_df = final_df[
    ~final_df['head_detail_name'].isna()
].reset_index(drop=True)
final_df
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 17,129 remaining


## 6. Standardise Fixed-Value Columns

In [26]:
final_df['head_type'] = 'ChemicalEntity'
final_df['tail_type'] = 'Mutation'
final_df['relation']  = 'ChemicalEntity_Mutation'
final_df['tail_id_is'] = 'Mutation'

# head_id_is: DB-prefix → Drugbank, numeric → Pubchem, else NaN
final_df['head'] = final_df['head'].astype(str)
final_df['head_id_is'] = np.where(
    final_df['head'].isna(), np.nan,
    np.where(final_df['head'].str.startswith('DB'), 'Drugbank',
    np.where(final_df['head'].str.match(r'^\d+$'), 'Pubchem', None)) # intentional None becauce some are just names like Cermides etc
)

print("Unique relation:",   set(final_df['relation']))
print("Unique head_type:",  set(final_df['head_type']))
print("Unique tail_type:",  set(final_df['tail_type']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'ChemicalEntity_Mutation'}
Unique head_type: {'ChemicalEntity'}
Unique tail_type: {'Mutation'}
Unique head_id_is: {'Pubchem', None, 'Drugbank'}
Unique kg_source: {'CKG', 'HALD_KG'}


In [27]:
# final_df[final_df['head_id_is'].isna()]

## 7. Deduplicate

In [28]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 13,808 rows


## 8. Add Schema Columns and Enforce Column Order

In [29]:
# EXTRA_COLS = ['head_smiles_name']
# final_df_dedup = final_df_dedup[REQUIRED_COLS + EXTRA_COLS]

# print(f"Final shape: {final_df_dedup.shape}")
# final_df_dedup.head()

In [30]:
# final_df_dedup[final_df_dedup['head_id_is'].isna()]

<!-- ## 9. QC — NaN Check -->

In [31]:

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
head_id_is,4320,0,4320
tail_id_is,0,0,0
head_detail_name,0,0,0


## 10. Save Output

In [32]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 13,808 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CHEMICALENTITY_MUTATION/ALL_CHEMICALENTITY_MUTATION.csv


In [12]:
#